# Ewe-Igbo-Yoruba (Yoruba New Testament only)

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import soundfile as sf
import io
import torch
import torchaudio
from datasets import load_dataset, Audio
from tqdm import tqdm
import IPython.display as ipd
from IPython.display import Markdown

# ── Fix: torchaudio 2.10 defaults to torchcodec which needs FFmpeg libs ───────
# Monkey-patch torchaudio.load to use soundfile (works for .wav files)
_original_torchaudio_load = torchaudio.load

def _torchaudio_load_sf(filepath, *args, **kwargs):
    data, samplerate = sf.read(filepath, dtype="float32")
    tensor = torch.from_numpy(data.T if data.ndim > 1 else data[None, :])
    return tensor, samplerate

torchaudio.load = _torchaudio_load_sf
print("Patched torchaudio.load → soundfile backend")

/home/mila/g/guzmand/scratch/.conda/envs/F5-TTS/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Patched torchaudio.load → soundfile backend


In [2]:
LANGUAGE = "Yoruba"

In [3]:
# ── Paths ──────────────────────────────────────────────────────────────────────
# Checkpoint & vocab  (edit CKPT_PATH to switch models)
CKPT_PATH    = "ckpts/F5TTS_v1_Base_vocos_custom_open-bible-ewe-igbo-yoruba/model_last.pt"
VOCAB_FILE   = "data/open-bible-ewe-igbo-yoruba_custom/vocab.txt"
MODEL_CFG    = "src/f5_tts/configs/F5TTS_v1_Base_Open_Bible_Ewe-Igbo-Yoruba.yaml"

OUTPUT_DIR   = "synthesis_output/F5TTS_v1_Base_vocos_custom_open-bible-ewe-igbo-yoruba"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load test set ──────────────────────────────────────────────────────────────
def get_audio_filename(example):
    # Now example["audio"] is {'path': '...', 'bytes': b'...'}
    example["filename"] = example["audio"]["path"].split("/")[-1]
    return example
    
ds = load_dataset(
    "parquet",
    data_files={"test": f"hf://datasets/davidguzmanr/open-bible-resources/{LANGUAGE}/test-*.parquet"},
    split="test"
)
ds = ds.cast_column("audio", Audio(decode=False))
ds = ds.map(get_audio_filename)

# Convert to pandas DataFrame and remove the audio column
test = ds.remove_columns("audio").to_pandas()
print(f"Validation samples: {len(test)}")
test.head()

Validation samples: 1532


,text,testament,book,chapter,verse,duration_seconds,speaker_id,filename
0,"Bí ó ṣe ń bá mi sọ̀rọ̀, mo ti sùn lọ fọnfọn, b...",Old Testament,Daniel,008,018,11.81,SPEAKER_00,DAN_008_Verse_018.wav
1,"Yẹra fún un, má ṣe rìn níbẹ̀; yàgò fún un kí o...",Old Testament,Proverbs,004,015,6.08,SPEAKER_00,PRO_004_Verse_015.wav
2,Ní ọjọ́ náà gan an ni Abrahamu mú Iṣmaeli ọmọ ...,Old Testament,Genesis,017,023,16.93,SPEAKER_00,GEN_017_Verse_023.wav
3,“ ‘Nítorí èyí ni Olúwa Olódùmarè wí: Èmi fúnra...,Old Testament,Ezekiel,034,011,9.73,SPEAKER_00,EZK_034_Verse_011.wav
4,"“Èmi ni Olúwa Ọlọ́run yín, tí ó mú un yín jáde...",Old Testament,Exodus,020,002,7.55,SPEAKER_00,EXO_020_Verse_002.wav


In [4]:
# ── Save ground-truth audio files ─────────────────────────────────────────────
GROUND_TRUTH_DIR = f"{OUTPUT_DIR}/ground-truth"
os.makedirs(GROUND_TRUTH_DIR, exist_ok=True)

for example in tqdm(ds, desc="Saving ground-truth WAVs"):
    filename = example["filename"]                  # e.g. "abc123.mp3"
    stem     = os.path.splitext(filename)[0]        # strip original extension
    out_path = f"{GROUND_TRUTH_DIR}/{stem}.wav"

    audio_bytes = example["audio"]["bytes"]         # raw bytes from the parquet
    
    # Read from bytes buffer → numpy array + sample rate
    with io.BytesIO(audio_bytes) as buf:
        audio_array, sample_rate = sf.read(buf)

    sf.write(out_path, audio_array, sample_rate)

print(f"Saved {len(ds)} WAV files to {GROUND_TRUTH_DIR}")

Saving ground-truth WAVs:  26%|██▋       | 404/1532 [00:15<00:43, 25.69it/s]


KeyboardInterrupt: 

In [ ]:
# ── Pick a reference audio from the training set ──────────────────────────────
# F5-TTS clones the speaker from a short reference clip + its transcript.
# We pick one training utterance (~7-9 s) as the universal reference for this
# single-speaker dataset. Change REF_INDEX to try a different prompt.
METADATA_PATH = "data/open-bible-ewe-igbo-yoruba/metadata.csv"
WAV_DIR = "data/open-bible-yoruba/wavs"

train = pd.read_csv(METADATA_PATH, sep="|")
train['language'] = train["audio_file"].apply(lambda x: x.split("/")[-3].split("-")[-1].capitalize())
train = train[train["language"] == LANGUAGE]
train["duration_seconds"] = train["audio_file"].apply(lambda path: sf.info(path).duration)
train.head()

# Pick a reference utterance with a "typical" duration (6-10 s)
candidates = train[(train["duration_seconds"] >= 6) & (train["duration_seconds"] <= 10)]
REF_INDEX = 1  # index into the filtered candidates; change to try others

ref_row = candidates.iloc[REF_INDEX]
REF_AUDIO = ref_row["audio_file"]
REF_TEXT  = ref_row["text"]

print(f"Reference audio : {REF_AUDIO}")
print(f"Reference text  : {REF_TEXT}")
print(f"Reference dur   : {ref_row['duration_seconds']:.2f}s")

ipd.display(ipd.Audio(REF_AUDIO))

Reference audio : /network/scratch/g/guzmand/Repositories/open-bible-models/F5-TTS/data/open-bible-yoruba/wavs/New-Testament-Acts-026-009.wav
Reference text  : “Èmi tilẹ̀ rò nínú ará mi nítòótọ́ pé, ó yẹ ki èmi ṣe ọ̀pọ̀lọpọ̀ ohun lòdì sí orúkọ Jesu tí Nasareti.
Reference dur   : 8.06s


In [ ]:
# ── Load model & vocoder ──────────────────────────────────────────────────────
from hydra.utils import get_class
from omegaconf import OmegaConf

from f5_tts.infer.utils_infer import (
    load_model,
    load_vocoder,
    preprocess_ref_audio_text,
    infer_process,
)

# Load config
model_cfg = OmegaConf.load(MODEL_CFG)
model_cls = get_class(f"f5_tts.model.{model_cfg.model.backbone}")
model_arc = model_cfg.model.arch

# Vocoder
vocoder = load_vocoder(vocoder_name="vocos", is_local=False)

# TTS model
ema_model = load_model(
    model_cls,
    model_arc,
    CKPT_PATH,
    mel_spec_type="vocos",
    vocab_file=VOCAB_FILE,
)
print("Model loaded ✓")

Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  data/open-bible-ewe-igbo-yoruba_custom/vocab.txt
token :  custom
model :  ckpts/F5TTS_v1_Base_vocos_custom_open-bible-ewe-igbo-yoruba/model_last.pt 

Model loaded ✓


In [ ]:
# ── Preprocess reference audio ─────────────────────────────────────────────────
ref_audio, ref_text = preprocess_ref_audio_text(REF_AUDIO, REF_TEXT)
print(f"Preprocessed ref_audio: {ref_audio}")
print(f"Preprocessed ref_text : {ref_text}")

Converting audio...
Using custom reference text...

ref_text   “Èmi tilẹ̀ rò nínú ará mi nítòótọ́ pé, ó yẹ ki èmi ṣe ọ̀pọ̀lọpọ̀ ohun lòdì sí orúkọ Jesu tí Nasareti. 
Preprocessed ref_audio: /tmp/tmpmrn1q9ll.wav
Preprocessed ref_text : “Èmi tilẹ̀ rò nínú ará mi nítòótọ́ pé, ó yẹ ki èmi ṣe ọ̀pọ̀lọpọ̀ ohun lòdì sí orúkọ Jesu tí Nasareti. 


In [ ]:
# ── Batch inference over the validation set ────────────────────────────────────
# For each row in the validation TSV, synthesize speech from the text column
# using the same reference audio. Outputs are saved as individual .wav files.

generated_files = []
errors = []

for idx, row in tqdm(test.head().iterrows(), total=len(test), desc="Synthesizing"):
    gen_text = row["text"]
    out_filename = row["filename"]  # keep original filename
    out_path = os.path.join(OUTPUT_DIR, out_filename)

    # Skip if already generated
    if os.path.exists(out_path):
        generated_files.append(out_path)
        continue

    try:
        audio_segment, final_sample_rate, spectrogram = infer_process(
            ref_audio,
            ref_text,
            gen_text,
            ema_model,
            vocoder,
            mel_spec_type="vocos",
        )
        sf.write(out_path, audio_segment, final_sample_rate)
        generated_files.append(out_path)
    except Exception as e:
        print(f"Error on {out_filename}: {e}")
        errors.append({"filename": out_filename, "error": str(e)})

print(f"\nDone! {len(generated_files)} files generated, {len(errors)} errors.")
print(f"Output directory: {OUTPUT_DIR}")

Synthesizing:   0%|          | 5/1532 [00:00<00:00, 2737.08it/s]


Done! 5 files generated, 0 errors.
Output directory: synthesis_output/F5TTS_v1_Base_vocos_custom_open-bible-ewe-igbo-yoruba


In [ ]:
# display(Markdown(f"""
# **Testament:** {sample['testament']}
# **Book:** {sample['book']}
# **Chapter:** {sample['chapter']}
# **Verse:** {sample['verse']}
# **Duration:** {sample['duration_seconds']}s

# > {sample['text']}
# """))

In [ ]:
# ── Quick sanity check: listen to a sample ─────────────────────────────────────
import IPython.display as ipd

ref_index = 0

sample_path = generated_files[0]
transcript = test.iloc[ref_index]["text"]

stem = os.path.splitext(test.iloc[ref_index]["filename"])[0]
gt_path = os.path.join(GROUND_TRUTH_DIR, f"{stem}.wav")

ipd.display(ipd.Markdown(f"""
**Transcript:** {transcript}

**Generated audio:**
"""))
ipd.display(ipd.Audio(sample_path))

ipd.display(ipd.Markdown("**Ground-truth audio:**"))
ipd.display(ipd.Audio(gt_path))


**Transcript:** Bí ó ṣe ń bá mi sọ̀rọ̀, mo ti sùn lọ fọnfọn, bí mo ṣe da ojú bolẹ̀. Nígbà náà ni ó fi ọwọ́ kàn mí, ó sì gbé mi dúró lórí ẹsẹ̀ mi.

**Generated audio:**


**Ground-truth audio:**

# Yoruba New Testament only

In [2]:
import os
import sys

import numpy as np
import pandas as pd
import soundfile as sf
import io
import torch
import torchaudio
from datasets import load_dataset, Audio
from tqdm import tqdm
import IPython.display as ipd
from IPython.display import Markdown

# ── Fix: torchaudio 2.10 defaults to torchcodec which needs FFmpeg libs ───────
# Monkey-patch torchaudio.load to use soundfile (works for .wav files)
_original_torchaudio_load = torchaudio.load

def _torchaudio_load_sf(filepath, *args, **kwargs):
    data, samplerate = sf.read(filepath, dtype="float32")
    tensor = torch.from_numpy(data.T if data.ndim > 1 else data[None, :])
    return tensor, samplerate

torchaudio.load = _torchaudio_load_sf
print("Patched torchaudio.load → soundfile backend")

Patched torchaudio.load → soundfile backend


In [3]:
LANGUAGE = "Yoruba"

In [4]:
# ── Paths ──────────────────────────────────────────────────────────────────────
# Checkpoint & vocab  (edit CKPT_PATH to switch models)
CKPT_PATH    = "ckpts/F5TTS_v1_Base_vocos_custom_open-bible-ewe-igbo-yoruba/model_last.pt"
VOCAB_FILE   = "data/open-bible-ewe-igbo-yoruba_custom/vocab.txt"
MODEL_CFG    = "src/f5_tts/configs/F5TTS_v1_Base_Open_Bible_Ewe-Igbo-Yoruba.yaml"

OUTPUT_DIR   = "synthesis_output/F5TTS_v1_Base_vocos_custom_open-bible-ewe-igbo-yoruba"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load test set ──────────────────────────────────────────────────────────────
def get_audio_filename(example):
    # Now example["audio"] is {'path': '...', 'bytes': b'...'}
    example["filename"] = example["audio"]["path"].split("/")[-1]
    return example
    
ds = load_dataset(
    "parquet",
    data_files={"test": f"hf://datasets/davidguzmanr/open-bible-resources/{LANGUAGE}/test-*.parquet"},
    split="test"
)
ds = ds.cast_column("audio", Audio(decode=False))
ds = ds.map(get_audio_filename)

# Convert to pandas DataFrame and remove the audio column
test = ds.remove_columns("audio").to_pandas()
print(f"Validation samples: {len(test)}")
test.head()

Validation samples: 1532


,text,testament,book,chapter,verse,duration_seconds,speaker_id,filename
0,"Bí ó ṣe ń bá mi sọ̀rọ̀, mo ti sùn lọ fọnfọn, b...",Old Testament,Daniel,008,018,11.81,SPEAKER_00,DAN_008_Verse_018.wav
1,"Yẹra fún un, má ṣe rìn níbẹ̀; yàgò fún un kí o...",Old Testament,Proverbs,004,015,6.08,SPEAKER_00,PRO_004_Verse_015.wav
2,Ní ọjọ́ náà gan an ni Abrahamu mú Iṣmaeli ọmọ ...,Old Testament,Genesis,017,023,16.93,SPEAKER_00,GEN_017_Verse_023.wav
3,“ ‘Nítorí èyí ni Olúwa Olódùmarè wí: Èmi fúnra...,Old Testament,Ezekiel,034,011,9.73,SPEAKER_00,EZK_034_Verse_011.wav
4,"“Èmi ni Olúwa Ọlọ́run yín, tí ó mú un yín jáde...",Old Testament,Exodus,020,002,7.55,SPEAKER_00,EXO_020_Verse_002.wav


In [5]:
# ── Save ground-truth audio files ─────────────────────────────────────────────
GROUND_TRUTH_DIR = f"{OUTPUT_DIR}/ground-truth"
os.makedirs(GROUND_TRUTH_DIR, exist_ok=True)

for example in tqdm(ds, desc="Saving ground-truth WAVs"):
    filename = example["filename"]                  # e.g. "abc123.mp3"
    stem     = os.path.splitext(filename)[0]        # strip original extension
    out_path = f"{GROUND_TRUTH_DIR}/{stem}.wav"

    audio_bytes = example["audio"]["bytes"]         # raw bytes from the parquet
    
    # Read from bytes buffer → numpy array + sample rate
    with io.BytesIO(audio_bytes) as buf:
        audio_array, sample_rate = sf.read(buf)

    sf.write(out_path, audio_array, sample_rate)

print(f"Saved {len(ds)} WAV files to {GROUND_TRUTH_DIR}")

Saving ground-truth WAVs: 100%|██████████| 1532/1532 [00:23<00:00, 66.25it/s]

Saved 1532 WAV files to synthesis_output/F5TTS_v1_Base_vocos_custom_open-bible-ewe-igbo-yoruba/ground-truth


In [ ]:
# ── Pick a reference audio from the training set ──────────────────────────────
# F5-TTS clones the speaker from a short reference clip + its transcript.
# We pick one training utterance (~7-9 s) as the universal reference for this
# single-speaker dataset. Change REF_INDEX to try a different prompt.
METADATA_PATH = "data/open-bible-ewe-igbo-yoruba/metadata.csv"
WAV_DIR = "data/open-bible-yoruba/wavs"

train = pd.read_csv(METADATA_PATH, sep="|")
train['language'] = train["audio_file"].apply(lambda x: x.split("/")[-3].split("-")[-1].capitalize())
train = train[train["language"] == LANGUAGE]
train["duration_seconds"] = train["audio_file"].apply(lambda path: sf.info(path).duration)
train.head()

# Pick a reference utterance with a "typical" duration (6-10 s)
candidates = train[(train["duration_seconds"] >= 6) & (train["duration_seconds"] <= 10)]
REF_INDEX = 1  # index into the filtered candidates; change to try others

ref_row = candidates.iloc[REF_INDEX]
REF_AUDIO = ref_row["audio_file"]
REF_TEXT  = ref_row["text"]

print(f"Reference audio : {REF_AUDIO}")
print(f"Reference text  : {REF_TEXT}")
print(f"Reference dur   : {ref_row['duration_seconds']:.2f}s")

ipd.display(ipd.Audio(REF_AUDIO))

In [ ]:
# ── Load model & vocoder ──────────────────────────────────────────────────────
from hydra.utils import get_class
from omegaconf import OmegaConf

from f5_tts.infer.utils_infer import (
    load_model,
    load_vocoder,
    preprocess_ref_audio_text,
    infer_process,
)

# Load config
model_cfg = OmegaConf.load(MODEL_CFG)
model_cls = get_class(f"f5_tts.model.{model_cfg.model.backbone}")
model_arc = model_cfg.model.arch

# Vocoder
vocoder = load_vocoder(vocoder_name="vocos", is_local=False)

# TTS model
ema_model = load_model(
    model_cls,
    model_arc,
    CKPT_PATH,
    mel_spec_type="vocos",
    vocab_file=VOCAB_FILE,
)
print("Model loaded ✓")

In [ ]:
# ── Preprocess reference audio ─────────────────────────────────────────────────
ref_audio, ref_text = preprocess_ref_audio_text(REF_AUDIO, REF_TEXT)
print(f"Preprocessed ref_audio: {ref_audio}")
print(f"Preprocessed ref_text : {ref_text}")

In [ ]:
# ── Batch inference over the validation set ────────────────────────────────────
# For each row in the validation TSV, synthesize speech from the text column
# using the same reference audio. Outputs are saved as individual .wav files.

generated_files = []
errors = []

for idx, row in tqdm(test.head().iterrows(), total=len(test), desc="Synthesizing"):
    gen_text = row["text"]
    out_filename = row["filename"]  # keep original filename
    out_path = os.path.join(OUTPUT_DIR, out_filename)

    # Skip if already generated
    if os.path.exists(out_path):
        generated_files.append(out_path)
        continue

    try:
        audio_segment, final_sample_rate, spectrogram = infer_process(
            ref_audio,
            ref_text,
            gen_text,
            ema_model,
            vocoder,
            mel_spec_type="vocos",
        )
        sf.write(out_path, audio_segment, final_sample_rate)
        generated_files.append(out_path)
    except Exception as e:
        print(f"Error on {out_filename}: {e}")
        errors.append({"filename": out_filename, "error": str(e)})

print(f"\nDone! {len(generated_files)} files generated, {len(errors)} errors.")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# ── Quick sanity check: listen to a sample ─────────────────────────────────────
import IPython.display as ipd

ref_index = 0

sample_path = generated_files[0]
transcript = test.iloc[ref_index]["text"]

stem = os.path.splitext(test.iloc[ref_index]["filename"])[0]
gt_path = os.path.join(GROUND_TRUTH_DIR, f"{stem}.wav")

ipd.display(ipd.Markdown(f"""
**Transcript:** {transcript}

**Generated audio:**
"""))
ipd.display(ipd.Audio(sample_path))

ipd.display(ipd.Markdown("**Ground-truth audio:**"))
ipd.display(ipd.Audio(gt_path))